# NeuroGraph-X · L3 + L4 · Baseline GNN on real ABIDE (CC200)
Owner: Laiba Afridi. Protocol: memo 0006 (amended 12 Oct) and memo 0007.

**Inputs (Add Input on the right):**
1. `neurographx-code`: the repository zip (`git archive`, with a COMMIT file).
2. The CC200 connectomes from Aleesha: `rois_cc200.npz` and `rois_cc200.json`.

**Settings:** Accelerator GPU T4, Internet On.

**Rule:** run the cross-validation cells as often as you like. Run the FINAL cell **once**.

In [ ]:
!pip install -q torch_geometric

## 1. Copy the code and the connectomes into a writable folder

In [ ]:
import glob, os, shutil
code_root = os.path.dirname(glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)[0])
REPO = "/kaggle/working/ngx"
if not os.path.exists(REPO):
    shutil.copytree(code_root, REPO)
dest = f"{REPO}/data/processed/connectomes/cpac_filt_global"
os.makedirs(dest, exist_ok=True)
for name in ("rois_cc200.npz", "rois_cc200.json"):
    src = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert src, f"{name} not found: add Aleesha's connectome dataset as an input"
    shutil.copy(src[0], dest)
os.chdir(REPO)
print("repo:", REPO, "| commit:", open("COMMIT").read().strip() if os.path.exists("COMMIT") else "unknown")
print(os.listdir(dest))

In [ ]:
!PYTHONPATH=src python -m pytest -q tests/test_leakage.py tests/test_gnn_data.py tests/test_run_logger.py tests/test_inference.py

Expected: all passed (none skipped here, because Kaggle has PyTorch).

## 2. Cross-validation (development pool only, the frozen set is not read)

In [ ]:
!PYTHONPATH=src python scripts/train_gnn.py --stage cv

Read the last lines: CV accuracy vs the majority baseline, balanced accuracy, the chosen number of epochs and the **run_id**.
Optional: compare settings, still on CV only, e.g. `--density 0.05` or `--density 0.2`. Pick the CV run you want for the final model.

In [ ]:
CV_RUN_ID = "PASTE_THE_CV_RUN_ID_HERE"

## 3. FINAL: train on the whole development pool, measure the frozen set ONCE

In [ ]:
!PYTHONPATH=src python scripts/train_gnn.py --stage final --epochs-from-cv {CV_RUN_ID}

## 4. Inference on one frozen participant (L7, demo step 6)

In [ ]:
import sys; sys.path.insert(0, "src")
import pandas as pd
from neurographx.m3_model.data import load_connectomes, make_get_graph
from neurographx.m3_model.inference import predict
from neurographx.m3_model.run_logger import get_model, get_run
entry = get_model("cc200"); run = get_run(entry["run_id"])
cset = load_connectomes("data/processed/connectomes", "rois_cc200")
get_graph = make_get_graph(cset, run["data_config"]["density"], run["data_config"]["fill_nan"])
sub = int(pd.read_csv("data/frozen/frozen_eval_100.csv").SUB_ID[0])
print(predict(f"{sub:07d}", "cc200", get_graph))

## 5. Evidence

In [ ]:
!tail -n 2 runs/run_log.jsonl
!cat runs/model_registry.json
!ls -la checkpoints

Save Version, then download from Output: `ngx/runs/run_log.jsonl`, `ngx/runs/model_registry.json` and the checkpoint `ngx/checkpoints/cc200_<run_id>.pt`.
Screenshot the CV summary, the frozen-set line and the inference output.